In [1]:
import pickle
import chromadb
import numpy as np

from utils.funcs import *
from tqdm.autonotebook import tqdm

/home/ytian/anaconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('models/embeddings.pkl', 'rb') as f:
  embeddings = pickle.load(f)
len(embeddings)

34099

In [3]:
len(embeddings[0])

320

In [4]:
tcr_ids = [f"tcr{i}" for i in range(len(embeddings))]

In [5]:
client = chromadb.PersistentClient(path="models")
collection = client.get_collection(name="tcrs")

In [6]:
search_embedding(collection, embeddings[0], 1)

{'ids': ['tcr0'],
 'metadatas': [{'Antigen Epitope': 'KLGGALQAK',
   'Antigen Protein': 'IE1',
   'Antigen Source': 'CMV',
   'CDR3.beta.aa': 'CASSPKTSVTYNEQFF',
   'Database': 'VDJdb',
   'Reference': 'https://www.10xgenomics.com/resources/application-notes/a-new-way-of-exploring-immunity-linking-highly-multiplexed-antigen-recognition-to-immune-repertoire-and-phenotype/#',
   'Species': 'Human',
   'TRBJ': 'TRBJ2-1*01',
   'TRBV': 'TRBV7-9*01'}],
 'distances': [-2.384185791015625e-07]}

In [7]:
def top_match_id(embedding):
    try:
        result = search_embedding(collection, [embedding], 1)
        pred_id = result['ids'][0]
    except:
        pred_id = np.nan
    return pred_id

In [8]:
preds = []
for embedding in tqdm(embeddings):
    preds.append(top_match_id(embedding))

len(preds)

  0%|          | 40/34099 [00:00<01:26, 392.34it/s]

100%|██████████| 34099/34099 [01:07<00:00, 508.80it/s]


34099

In [9]:
accuracy = (np.array(tcr_ids) == np.array(preds)).mean()
print(f"Accuracy: {accuracy*100:.1f}%")

Accuracy: 99.7%
